In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Custom functions
from utils.visualisation_functions import plot_sankey_hierarchy
from utils.data_manipulations import build_activity_name, add_site_id
from core.lci_database_builder import LCIDatabaseBuilder
from utils.conversion_functions import map_technosphere_to_ecoinvent, map_biosphere_to_ecoinvent
from utils.constants import CA_provinces
from utils.data_manipulations import add_land_substance_name

In [ ]:
import bw2data as bd
from bw2io import BW2Package
import brightway2 as bw

# Plot selected sites

In [3]:
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [4]:
production_df_plot = production_df[production_df['Create_LCI?'] == 'Yes']
columns_to_plot = ['province', 'mining_processing_type', 'archetypes', 'Stream']

In [5]:
# plot_sankey_hierarchy(production_df_plot, columns_to_plot,
#                       html_output="data/MetalliCan/sites_for_lci_archetypes.html",
#                       output_image="data/MetalliCan/site_selection_sankey")

# Import cleaned MetalliCan data

In [6]:
# Market share
df_market = pd.read_excel(r'data/SI/SI_2_site_selection.xlsx', sheet_name='market_shares')

In [7]:
# Pre-processed production table
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [8]:
# Add activitiy_name to production_df and site_id
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)
production_df = add_site_id(production_df)

In [9]:
# Normalized MetalliCan tables per ore processed
energy_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv')
material_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv')
biosphere_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv')
land_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv')
carbon_stock_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/carbon_stock_df.csv')

In [10]:
energy_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/energy_df.csv')
material_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/material_df.csv')
biosphere_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/biosphere_df.csv')
land_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/land_df.csv')
carbon_stock_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/carbon_stock_df.csv')

In [11]:
# Removing rows with value_normalized is NaN in the biosphere dfs
#biosphere_ore_df = biosphere_ore_df[~biosphere_ore_df['value_normalized'].isna()]
#biosphere_stream_df['value_normalized'] = biosphere_stream_df['value_normalized'] / 1e6

In [12]:
# Drop rows where 'unit' = 'ha'
biosphere_stream_df = biosphere_stream_df[biosphere_stream_df['unit'] != 'ha']
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_stream_df_bgf['unit'] != 'ha']

In [13]:
land_stream_df = add_land_substance_name(land_stream_df)
land_stream_df_bgf = add_land_substance_name(land_stream_df_bgf)

In [14]:
carbon_stock_stream_df.rename(columns={"flow_type": "substance_name"}, inplace=True)
carbon_stock_stream_df_bgf.rename(columns={"flow_type": "substance_name"}, inplace=True)

# Keeping only relevant columns

In [15]:
nrj_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
material_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
biosphere_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized']

In [16]:
energy_stream_df = energy_stream_df[nrj_col]
material_stream_df = material_stream_df[material_col]
biosphere_stream_df = biosphere_stream_df[biosphere_col]
land_stream_df = land_stream_df[biosphere_col]

In [17]:
energy_stream_df_bgf = energy_stream_df_bgf[nrj_col]
material_stream_df_bgf = material_stream_df_bgf[material_col]
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_col]
land_stream_df_bgf = land_stream_df_bgf[biosphere_col]

In [18]:
# Put energy_df and material_df together, and biosphere_df and land_df together
technosphere_stream_df = pd.concat([energy_stream_df, material_stream_df], ignore_index=True)
biosphere_stream_df = pd.concat([biosphere_stream_df, land_stream_df, carbon_stock_stream_df], ignore_index=True)
technosphere_stream_df_bgf = pd.concat([energy_stream_df_bgf, material_stream_df_bgf], ignore_index=True)
biosphere_stream_df_bgf = pd.concat([biosphere_stream_df_bgf, land_stream_df_bgf, carbon_stock_stream_df_bgf], ignore_index=True)

In [19]:
# Remove rows where value_normalized is NaN
technosphere_stream_df = technosphere_stream_df[~technosphere_stream_df['value_normalized'].isna()]
technosphere_stream_df_bgf = technosphere_stream_df_bgf[~technosphere_stream_df_bgf['value_normalized'].isna()]

In [20]:
# Add the province from the main_df to specify electricity location later
technosphere_stream_df = technosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df = biosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
technosphere_stream_df_bgf = technosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df_bgf = biosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')

# Map MetalliCan flows to EI and RI flows

## Technosphere flows

In [21]:
mapping_technosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='technosphere')
mapping_technosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='technosphere')

In [22]:
# Apply the function
mapped_technosphere_ri_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ei, CA_provinces)

⚠️ Les flux suivants n'ont pas trouvé de correspondance dans Ecoinvent:
 - Acids
 - Soda ash
 - Reagants
 - Tailings|Other
⚠️ Les flux suivants n'ont pas trouvé de correspondance dans Ecoinvent:
 - Acids
 - Soda ash
 - Reagants
 - Tailings|Other


In [23]:
mapped_technosphere_ri_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ei, CA_provinces)

In [24]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_technosphere_ri_df = mapped_technosphere_ri_df[(mapped_technosphere_ri_df["Activity"] != "No mapping")]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[(mapped_technosphere_ei_df["Activity"] != "No mapping")]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[(mapped_technosphere_ri_df_bgf["Activity"] != "No mapping")]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[(mapped_technosphere_ei_df_bgf["Activity"] != "No mapping")]

In [25]:
technosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Amount_min', 'Amount_mean', 'Amount_max', 'Activity', 'Product', 'Unit', 'Location', 'Database']

In [26]:
mapped_technosphere_ri_df = mapped_technosphere_ri_df[technosphere_col_for_lci]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[technosphere_col_for_lci]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[technosphere_col_for_lci]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[technosphere_col_for_lci]

## Biosphere flows mapping

In [27]:
mapping_biosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='biosphere')
mapping_biosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='biosphere')

In [28]:
mapped_biosphere_ri_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ei, CA_provinces)
mapped_biosphere_ri_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ei, CA_provinces)

Index(['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit',
       'value_normalized', 'normalization_key', 'allocation_factor',
       'mining_processing_type', 'archetypes', 'data_source',
       'reference_mass_unit', 'province', 'Type', 'substance_id',
       'compartment_name', 'release_pathway', 'flow_direction',
       'MetalliCan_unit', 'DB_to_map', 'Flow name', 'Compartments', 'Unit',
       'Comment', 'Alternatives'],
      dtype='object')
⚠️ 1 biosphere flows could not be mapped to Ecoinvent:
   - PFCs
⚠️ No conversion defined for tco2eq → nan (flow: PFCs)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter (flow: Water)
⚠️ No conversion defined for kg → cubic meter 

In [29]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_biosphere_ri_df = mapped_biosphere_ri_df[(mapped_biosphere_ri_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df["Amount"].isna())]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[(mapped_biosphere_ei_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df["Amount"].isna())]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[(mapped_biosphere_ri_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df_bgf["Amount"].isna())]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[(mapped_biosphere_ei_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df_bgf["Amount"].isna())]

In [30]:
biosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Unit', 'Flow Name', 'Compartments', 'Database']

In [31]:
mapped_biosphere_ri_df = mapped_biosphere_ri_df[biosphere_col_for_lci]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[biosphere_col_for_lci]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[biosphere_col_for_lci]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[biosphere_col_for_lci]

In [32]:
# Add province and commodities from NRCan to production table
mapped_biosphere_ri_df = mapped_biosphere_ri_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df = mapped_biosphere_ei_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')

# LCI creation

## Regioinvent

In [33]:
# # Step 1 — initialize the builder
builder_regio = LCIDatabaseBuilder(db_name='metallican_lci_ri', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_lci_ri' created.


In [34]:
# # Step 2 — create the activity shells from the main dataframe
builder_regio.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_regio.lcis))

✅ Created 59 base LCI activities with production exchanges.
59


In [35]:
# # Step 3a — Populate with the technosphere exchanges
builder_regio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ri_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 218246 activities from Regioinvent
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff regionalized
✅ Added 885 technosphere exchanges.


In [36]:
# # Step 3b — Populate with the biosphere exchanges
builder_regio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ri_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
   ✅ Cached 110559 biosphere flows from biosphere3_spatialized_flows
✅ Added 13284 biosphere exchanges.


In [37]:
# # Step 4 - Consolidate duplicate flows
builder_regio.consolidate_exchanges()

🧮 Consolidation: 14228 → 2046 exchanges (summed duplicates).


In [38]:
builder_regio.build_market_activities(df_market)

🧩 Created 6 market activities.


6

In [39]:
builder_regio.write_to_database()

🧱 Writing 65 activities to database 'metallican_lci_ri'...
✅ Created 59 site-specific activities.
✅ Created 6 market activities.
✅ Saved exchanges: approx 2100 (failures: 0)
✅ Database 'metallican_lci_ri' processed successfully with 65 activities.


## Ecoinvent

In [40]:
# # Step 1 — initialize the builder
builder_ei = LCIDatabaseBuilder(db_name='metallican_lci_ei', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_lci_ei' created.


In [41]:
# # Step 2 — create the activity shells from the main dataframe
builder_ei.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_ei.lcis))

✅ Created 59 base LCI activities with production exchanges.
59


In [42]:
# # Step 3a — Populate with the technosphere exchanges
builder_ei.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 885 technosphere exchanges.


In [43]:
# # Step 3b — Populate with the biosphere exchanges
builder_ei.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13284 biosphere exchanges.


In [44]:
# # Step 4 - Consolidate duplicate flows
builder_ei.consolidate_exchanges()

🧮 Consolidation: 14228 → 2009 exchanges (summed duplicates).


In [45]:
builder_ei.build_market_activities(df_market)

🧩 Created 6 market activities.


6

In [46]:
builder_ei.write_to_database()

🧱 Writing 65 activities to database 'metallican_lci_ei'...
✅ Created 59 site-specific activities.
✅ Created 6 market activities.
✅ Saved exchanges: approx 2063 (failures: 0)
✅ Database 'metallican_lci_ei' processed successfully with 65 activities.


In [47]:
import brightway2 as bw

db = bw.Database("metallican_lci_ri")
mkt = db.get("market_Cu_concentrate")   # ou ton code exact

rows = []
for exc in mkt.technosphere():
    rows.append({
        "supplier_code": exc.input.get("code"),
        "supplier_name": exc.input.get("name"),
        "supplier_location": exc.input.get("location"),
        "share": exc["amount"],
    })

rows = sorted(rows, key=lambda d: d["share"], reverse=True)

print("n inputs:", len(rows))
for r in rows[:20]:
    print(f"{r['share']:.6f} | {r['supplier_name']} | {r['supplier_location']} | {r['supplier_code']}")



n inputs: 11
0.363552 | Open-pit mining and beneficiation at Highland Valley | CA-BC | BC-MAIN-bf503b6b_Cu concentrate
0.197434 | Open-pit mining and beneficiation at Gibraltar | CA-BC | BC-MAIN-6b4800fe_Cu concentrate
0.099622 | Open-pit mining and beneficiation at Mount Milligan | CA-BC | BC-MAIN-ed23117f_Cu concentrate
0.080237 | Underground mining and beneficiation at Kidd Creek | CA-ON | ON-MAIN-f8313ebd_Cu concentrate
0.076333 | Underground mining and beneficiation at New Afton | CA-BC | BC-MAIN-aa76f6f2_Cu concentrate
0.067633 | Open-pit mining and beneficiation at Copper Mountain | CA-BC | BC-MAIN-599152a0_Cu concentrate
0.048545 | Open-pit mining and beneficiation at Mount Polley | CA-BC | BC-MAIN-3f490561_Cu concentrate
0.043151 | Underground mining and beneficiation at Snow Lake | CA-MB | GRP-a13779f8_Cu concentrate
0.014201 | Open-pit mining and beneficiation at Red Chris | CA-BC | BC-MAIN-8eb8be0d_Cu concentrate
0.009153 | Underground mining and beneficiation at LaRonde | 

In [48]:
s = sum(exc["amount"] for exc in mkt.technosphere())
print("sum shares:", s)

sum shares: 1.0


In [49]:
exc = list(mkt.technosphere())[0]
print(exc.as_dict())


{'output': ('metallican_lci_ri', 'market_Cu_concentrate'), 'input': ('metallican_lci_ri', 'BC-MAIN-599152a0_Cu concentrate'), 'amount': 0.0676334699534121, 'type': 'technosphere'}


# Before and after gap filling

### Before gap filling - technosphere only

In [50]:
# Step 1 — initialize the builder
builder_ei_bgf_tech = LCIDatabaseBuilder(db_name='metallican_bgf_tech', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_bgf_tech' created.


In [51]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_tech.build_lci_entries(df=mapped_biosphere_ei_df_bgf)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 58 base LCI activities with production exchanges.
58


In [52]:
# Step 3a — Populate with the technosphere exchanges
builder_ei_bgf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df_bgf)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 258 technosphere exchanges.


In [53]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_bgf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [54]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_tech.consolidate_exchanges()

🧮 Consolidation: 316 → 244 exchanges (summed duplicates).


In [55]:
builder_ei_bgf_tech.write_to_database()

🧱 Writing 58 activities to database 'metallican_bgf_tech'...
✅ Created 58 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 244 (failures: 0)
✅ Database 'metallican_bgf_tech' processed successfully with 58 activities.


### Before gap filling - biosphere only

In [56]:
# Step 1 — initialize the builder
builder_ei_bgf_bio = LCIDatabaseBuilder(db_name='metallican_bgf_bio', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_bgf_bio' created.


In [57]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df_bgf)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 58 base LCI activities with production exchanges.
58


In [58]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [59]:
# Step 3b — Populate with the biosphere exchanges
builder_ei_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df_bgf)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13211 biosphere exchanges.


In [60]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 13269 → 1300 exchanges (summed duplicates).


In [61]:
builder_ei_bgf_bio.write_to_database()

🧱 Writing 58 activities to database 'metallican_bgf_bio'...
✅ Created 58 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 1300 (failures: 0)
✅ Database 'metallican_bgf_bio' processed successfully with 58 activities.


### After gap filling - technosphere only

In [62]:
# Step 1 — initialize the builder
builder_ei_agf_tech = LCIDatabaseBuilder(db_name='metallican_agf_tech', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_agf_tech' created.


In [63]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_agf_tech.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 59 base LCI activities with production exchanges.
58


In [64]:
# Step 3a — Populate with the technosphere exchanges
builder_ei_agf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 885 technosphere exchanges.


In [65]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_agf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [66]:
# Step 4 - Consolidate duplicate flows
builder_ei_agf_tech.consolidate_exchanges()

🧮 Consolidation: 944 → 722 exchanges (summed duplicates).


In [67]:
builder_ei_agf_tech.write_to_database()

🧱 Writing 59 activities to database 'metallican_agf_tech'...
✅ Created 59 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 722 (failures: 0)
✅ Database 'metallican_agf_tech' processed successfully with 59 activities.


### After gap filling - biosphere only

In [68]:
# Step 1 — initialize the builder
builder_ei_bgf_bio = LCIDatabaseBuilder(db_name='metallican_agf_bio', project_name='metallican_new')

📂 Active Brightway project: metallican_new
🆕 Database 'metallican_agf_bio' created.


In [69]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 59 base LCI activities with production exchanges.
58


In [70]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [71]:
# Step 3b — Populate with the biosphere exchanges
builder_ei_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13284 biosphere exchanges.


In [72]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 13343 → 1346 exchanges (summed duplicates).


In [73]:
builder_ei_bgf_bio.write_to_database()

🧱 Writing 59 activities to database 'metallican_agf_bio'...
✅ Created 59 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 1346 (failures: 0)
✅ Database 'metallican_agf_bio' processed successfully with 59 activities.


# Exports databases

In [ ]:
bd.projects.set_current("metallican_new")

In [75]:
fp_ei = BW2Package.export_obj(
    bd.Database("metallican_lci_ei"),
    filename="metallican_lci_ei",
    folder=r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases",
)
print("Exported to:", fp_ei)

Exported to: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ei.22c678a104a09ee0f51ac9ef9c9bfd21.bw2package


In [76]:
fp_ri = BW2Package.export_obj(
    bd.Database("metallican_lci_ri"),
    filename="metallican_lci_ri",
    folder=r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases",
)
print("Exported to:", fp_ri)

Exported to: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ri.d99527f2f15c54371f8254f102321b4c.bw2package


In [80]:
export_bw_database_to_excel("metallican_lci_ei", r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ei.xlsx")

Wrote: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ei.xlsx


(             database                             code  \
 0   metallican_lci_ei  ON-MAIN-52224e1e_Ni concentrate   
 1   metallican_lci_ei  QC-MAIN-c0660aec_Cu concentrate   
 2   metallican_lci_ei            QC-MAIN-b86f7d07_Doré   
 3   metallican_lci_ei  ON-MAIN-28f3f0fc_Ni concentrate   
 4   metallican_lci_ei            ON-MAIN-0aadf28f_Doré   
 ..                ...                              ...   
 60  metallican_lci_ei  BC-MAIN-3f490561_Cu concentrate   
 61  metallican_lci_ei            market_Cu_concentrate   
 62  metallican_lci_ei            QC-MAIN-c0660aec_Doré   
 63  metallican_lci_ei  QC-MAIN-e51eda66_Cu concentrate   
 64  metallican_lci_ei      GRP-a13779f8_Cu concentrate   
 
                                                  name reference_product  \
 0                     Underground mining at Creighton    Ni concentrate   
 1      Underground mining and beneficiation at Goldex    Cu concentrate   
 2   Open-pit and underground mining and beneficiat...        

In [78]:
export_bw_database_to_excel("metallican_lci_ri", r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ri.xlsx")

Wrote: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ri.xlsx


(             database                             code  \
 0   metallican_lci_ri  ON-MAIN-52224e1e_Ni concentrate   
 1   metallican_lci_ri            ON-MAIN-0aadf28f_Doré   
 2   metallican_lci_ri            QC-MAIN-9de9bb0d_Doré   
 3   metallican_lci_ri            market_Cu_concentrate   
 4   metallican_lci_ri  QC-MAIN-5ce331b8_Ni concentrate   
 ..                ...                              ...   
 60  metallican_lci_ri            ON-MAIN-1f126a43_Doré   
 61  metallican_lci_ri       GRP-91aaa60b_U concentrate   
 62  metallican_lci_ri            QC-MAIN-f9e41c2a_Doré   
 63  metallican_lci_ri            QC-MAIN-e51eda66_Doré   
 64  metallican_lci_ri            QC-MAIN-6dc537e6_Doré   
 
                                                  name reference_product  \
 0                     Underground mining at Creighton    Ni concentrate   
 1   Open-pit and underground mining and beneficiat...              Doré   
 2          Open-pit mining and beneficiation at Kiena        